# Qwen3.5-4B Fine-Tuning On FineTree (A100)

This notebook fine-tunes `Qwen/Qwen3.5-4B` using Unsloth on `asafd60/FineTree-annotated-pages`.

It mirrors the style of Unsloth's professional Qwen notebook flow:
- install + environment checks
- model + LoRA setup
- dataset prep
- training
- optional save/push
- manual evaluation


## Runtime

Recommended runtime:
- GPU: A100 40GB/80GB
- Python: 3.10+
- Internet access for HF model/dataset download

Dataset split behavior in this notebook:
- Loads the public dataset from Hugging Face
- Concatenates any existing splits
- Re-splits locally to `train/validation = 90/10`


In [ ]:
%%capture
import importlib.util
import os

!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None:
    !uv pip install -qqq "torch>=2.7,<2.10" torchvision

# Keep dependency style close to Unsloth professional notebooks.
!uv pip install -qqq --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install -qqq --upgrade datasets transformers accelerate bitsandbytes peft pillow matplotlib huggingface_hub

# Notebook-style Unsloth refresh path.
!uv pip install -qqq --upgrade --no-deps   "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"   "unsloth[base] @ git+https://github.com/unslothai/unsloth"


In [ ]:
import random
from pathlib import Path

import torch

SEED = 3407
MODEL_ID = "Qwen/Qwen3.5-4B"
DATASET_ID = "asafd60/FineTree-annotated-pages"
VAL_RATIO = 0.1

MAX_SEQ_LENGTH = 4096
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACC_STEPS = 16
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 2
WARMUP_RATIO = 0.03
LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 100

OUTPUT_DIR = "artifacts/qwen35-4b-finetree"
PUSH_ADAPTER_TO_HUB = False
HF_ADAPTER_REPO_ID = ""

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


## Load Model (Vision-first, text fallback)

The notebook prefers multimodal training if the model/runtime supports `FastVisionModel`.
If not available, it falls back to `FastLanguageModel` and trains only on instruction/output text.


In [ ]:
from unsloth import FastLanguageModel

use_vision_path = True
try:
    from unsloth import FastVisionModel, UnslothVisionDataCollator
except Exception:
    use_vision_path = False

if use_vision_path:
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else None,
        load_in_4bit=True,
        trust_remote_code=True,
    )
    model = FastVisionModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_rslora=False,
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        finetune_vision_layers=True,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
    )
    print("Using multimodal path (FastVisionModel).")
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else None,
        load_in_4bit=True,
        trust_remote_code=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=SEED,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )
    print("Using text fallback path (FastLanguageModel).")


In [ ]:
from datasets import DatasetDict, concatenate_datasets, load_dataset

raw = load_dataset(DATASET_ID)
if isinstance(raw, DatasetDict):
    all_splits = [raw[name] for name in raw.keys()]
    full_ds = concatenate_datasets(all_splits) if len(all_splits) > 1 else all_splits[0]
else:
    full_ds = raw

required_cols = {"image", "instruction", "output"}
missing = required_cols - set(full_ds.column_names)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

full_ds = full_ds.shuffle(seed=SEED)
split = full_ds.train_test_split(test_size=VAL_RATIO, seed=SEED)
train_ds = split["train"]
val_ds = split["test"]

print("Total rows:", len(full_ds))
print("Train rows:", len(train_ds))
print("Validation rows:", len(val_ds))


## Build Training Samples

Multimodal path:
- user = image + instruction
- assistant = output

Text fallback path:
- user = instruction
- assistant = output


In [ ]:
def build_messages(example):
    if use_vision_path:
        return {
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": example["image"]},
                        {"type": "text", "text": example["instruction"]},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": example["output"]}],
                },
            ]
        }
    return {
        "conversations": [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]},
        ]
    }

train_prepared = train_ds.map(build_messages)
val_prepared = val_ds.map(build_messages)

if not use_vision_path:
    def formatting_prompts_func(examples):
        convos = examples["conversations"]
        texts = [
            tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False,
            )
            for convo in convos
        ]
        return {"text": texts}

    train_prepared = train_prepared.map(formatting_prompts_func, batched=True)
    val_prepared = val_prepared.map(formatting_prompts_func, batched=True)

print("Prepared train rows:", len(train_prepared))
print("Prepared val rows:", len(val_prepared))


In [ ]:
from trl import SFTConfig, SFTTrainer

if use_vision_path:
    # Compatibility helper across Unsloth versions.
    def build_vision_collator(collator_cls, model, tokenizer):
        import inspect

        try:
            params = inspect.signature(collator_cls).parameters
        except Exception:
            params = {}

        kwargs = {}
        if "model" in params:
            kwargs["model"] = model
        if "tokenizer" in params:
            kwargs["tokenizer"] = tokenizer
        elif "processor" in params:
            kwargs["processor"] = tokenizer

        if kwargs:
            return collator_cls(**kwargs)

        for fallback_kwargs in (
            {"model": model, "tokenizer": tokenizer},
            {"model": model},
            {"processor": tokenizer},
            {"tokenizer": tokenizer},
            {},
        ):
            try:
                return collator_cls(**fallback_kwargs)
            except TypeError:
                continue
        raise RuntimeError("Failed to initialize UnslothVisionDataCollator.")

    data_collator = build_vision_collator(UnslothVisionDataCollator, model, tokenizer)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=data_collator,
        train_dataset=train_prepared,
        eval_dataset=val_prepared,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        args=SFTConfig(
            output_dir=OUTPUT_DIR,
            seed=SEED,
            per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACC_STEPS,
            learning_rate=LEARNING_RATE,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            optim="adamw_8bit",
            logging_steps=LOGGING_STEPS,
            eval_strategy="steps",
            eval_steps=EVAL_STEPS,
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=2,
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=False,
            report_to="none",
            remove_unused_columns=False,
        ),
    )
else:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_prepared,
        eval_dataset=val_prepared,
        args=SFTConfig(
            output_dir=OUTPUT_DIR,
            seed=SEED,
            dataset_text_field="text",
            per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACC_STEPS,
            learning_rate=LEARNING_RATE,
            num_train_epochs=NUM_TRAIN_EPOCHS,
            warmup_ratio=WARMUP_RATIO,
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            optim="adamw_8bit",
            logging_steps=LOGGING_STEPS,
            eval_strategy="steps",
            eval_steps=EVAL_STEPS,
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=2,
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=False,
            report_to="none",
            remove_unused_columns=False,
        ),
    )

print("Trainer ready.")


In [ ]:
if not use_vision_path:
    from unsloth.chat_templates import train_on_responses_only

    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user
",
        response_part="<|im_start|>assistant
",
    )
    print("Applied response-only loss masking.")
else:
    print("Response-only masking skipped in multimodal path.")


In [ ]:
# Start training
if use_vision_path:
    FastVisionModel.for_training(model)
else:
    FastLanguageModel.for_training(model)

train_result = trainer.train()
print(train_result)


In [ ]:
adapter_dir = Path(OUTPUT_DIR) / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved adapter:", adapter_dir)


In [ ]:
# Optional adapter push
if PUSH_ADAPTER_TO_HUB:
    if not HF_ADAPTER_REPO_ID:
        raise ValueError("Set HF_ADAPTER_REPO_ID before enabling push.")
    model.push_to_hub(HF_ADAPTER_REPO_ID)
    tokenizer.push_to_hub(HF_ADAPTER_REPO_ID)
    print("Pushed:", HF_ADAPTER_REPO_ID)
else:
    print("Skipping push (PUSH_ADAPTER_TO_HUB=False).")


## Manual Evaluation (Random Validation Sample)

This section displays a random image and prints:
- instruction
- model output
- ground truth output

BBox matching is intentionally not enforced here (manual qualitative check only).


In [ ]:
import matplotlib.pyplot as plt

sample_idx = random.randrange(len(val_ds))
sample = val_ds[sample_idx]

img = sample["image"]
plt.figure(figsize=(8, 10))
plt.imshow(img)
plt.axis("off")
plt.title(f"Validation sample #{sample_idx}")
plt.show()

print("=== Instruction ===")
print(sample["instruction"])

if use_vision_path:
    FastVisionModel.for_inference(model)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": sample["instruction"]},
            ],
        }
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(
        images=[img],
        text=text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.0,
        do_sample=False,
    )
    prediction = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
else:
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": sample["instruction"]}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.0,
        do_sample=False,
    )
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("
=== Model Output ===")
print(prediction)

print("
=== Ground Truth Output ===")
print(sample["output"])
